[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ImagingDataCommons/CloudSegmentator/blob/main/workflows/MOOSE/Notebooks/mooseInferenceNotebook.ipynb)

# **MOOSE Inference (Task 1): Download DICOM from IDC, convert to NIfTI, run moosez segmentation**

This notebook is the GPU-side of the MOOSE twoVM workflow. It:
1. Downloads DICOM series from Imaging Data Commons
2. Converts each series to NIfTI via dcm2niix
3. Runs moosez inference with the requested clinical CT models
4. Tars + lz4-compresses the moosez output tree as `moose_segmentations.tar.lz4`

Post-processing (DICOM-SEG generation) happens in Task 2 on a CPU-only VM.

Please cite:

Shiyam Sundar LK, et al. Fully automated, semantic segmentation of whole-body 18F-FDG PET/CT images based on data-centric artificial intelligence. J Nucl Med. 2022. https://doi.org/10.2967/jnumed.122.264063

Isensee, F., Jaeger, P.F., Kohl, S.A.A. et al. nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nat Methods 18, 203-211 (2021). https://doi.org/10.1038/s41592-020-01008-z

Li X, Morgan PS, Ashburner J, Smith J, Rorden C. (2016) The first step for neuroimaging data analysis: DICOM to NIfTI conversion. J Neurosci Methods. 264:47-56.

## Imports

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

import yaml
from idc_index.index import IDCClient

# ── wall-clock anchor ──────────────────────────────────────────────────────
NOTEBOOK_START = time.time()

def _elapsed(since=None):
    """Return seconds since `since` (default: NOTEBOOK_START) as a string."""
    return f"{time.time() - (since if since is not None else NOTEBOOK_START):.1f}s"

def _dir_size_mb(p: Path) -> float:
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / (1024 ** 2)

# ── environment snapshot ───────────────────────────────────────────────────
print(f"Python        : {sys.version}")
print(f"Working dir   : {os.getcwd()}")
print(f"Disk (/)      : ", end="")
_df = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
print(_df.stdout.splitlines()[-1] if _df.returncode == 0 else "unavailable")

try:
    import importlib.metadata as _im
    for _pkg in ("moosez", "nnunetv2", "SimpleITK", "torch", "idc_index", "papermill"):
        try:
            print(f"  {_pkg:<20} {_im.version(_pkg)}")
        except Exception:
            print(f"  {_pkg:<20} (not installed)")
except Exception as _e:
    print(f"Version check failed: {_e}")

print(f"\n[T+{_elapsed()}] Imports complete")


## Parameters

The cell below is tagged `parameters` so papermill can inject values at runtime.
When running interactively, edit the values directly.

In [ ]:
# Papermill injects SeriesInstanceUIDs as a Python list when called with
#   papermill notebook.ipynb out.ipynb -y "SeriesInstanceUIDs: [uid1, uid2]"
SeriesInstanceUIDs = [
    "1.3.6.1.4.1.14519.5.2.1.7009.9004.100143549999116733615345241533"
]

# Comma-separated list of moosez clinical CT model names
# Available: clin_ct_body, clin_ct_body_composition, clin_ct_cardiac,
#            clin_ct_digestive, clin_ct_lungs, clin_ct_muscles,
#            clin_ct_organs, clin_ct_peripheral_bones, clin_ct_ribs,
#            clin_ct_vertebrae
moose_models = "clin_ct_body,clin_ct_body_composition,clin_ct_cardiac,clin_ct_digestive,clin_ct_lungs,clin_ct_muscles,clin_ct_organs,clin_ct_peripheral_bones,clin_ct_ribs,clin_ct_vertebrae"

# 'cuda' for GPU, 'cpu' for CPU-only
accelerator = "cuda"


## Normalize parameters

In [ ]:
import re


def _flatten_uids(raw):
    """Normalize any papermill input shape (str / list / dict) to a clean
    list of SeriesInstanceUIDs.

    Tolerates malformed YAML where UIDs are separated by whitespace rather
    than commas, e.g. `[uid1, uid2 uid3 uid4]` — YAML parses the third item
    as one long string, so we re-split on whitespace+commas and drop
    anything that isn't a UID-shaped token (digits and dots only).
    """
    if isinstance(raw, str):
        try:
            parsed = yaml.safe_load(raw)
        except Exception:
            parsed = raw
        if isinstance(parsed, dict):
            parsed = parsed.get("SeriesInstanceUIDs", list(parsed.values()))
        raw = parsed
    items = list(raw) if isinstance(raw, (list, tuple)) else [raw]

    flat = []
    for item in items:
        if item is None:
            continue
        flat.extend(p for p in re.split(r"[\s,]+", str(item).strip()) if p)

    cleaned = [u for u in flat if re.fullmatch(r"[\d.]+", u)]
    seen, deduped = set(), []
    for u in cleaned:
        if u not in seen:
            seen.add(u)
            deduped.append(u)
    return deduped


series_uids = _flatten_uids(SeriesInstanceUIDs)
if not series_uids:
    raise ValueError(
        f"No valid SeriesInstanceUIDs parsed from input: {SeriesInstanceUIDs!r}"
    )

models = [m.strip() for m in moose_models.split(",") if m.strip()]

print(f"Series to process : {len(series_uids)}")
for u in series_uids:
    print(f"  {u}")
print(f"MOOSE models      : {models}")
print(f"Accelerator       : {accelerator}")

## GPU availability check

If `accelerator=cuda` is requested but no GPU is present, fall back to CPU rather than failing.

In [ ]:
usage_metrics = {"gpu": [], "series": {}}

# ── PyTorch / CUDA info ────────────────────────────────────────────────────
_t_torch = time.time()
try:
    import torch
    print(f"PyTorch version : {torch.__version__}")
    print(f"CUDA available  : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version    : {torch.version.cuda}")
        print(f"cuDNN version   : {torch.backends.cudnn.version()}")
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            free_b, total_b = torch.cuda.mem_get_info(i)
            print(f"  GPU {i}: {props.name}  |  "
                  f"VRAM total {total_b/1024**3:.1f} GB  free {free_b/1024**3:.1f} GB  |  "
                  f"SM {props.major}.{props.minor}  multiprocessors {props.multi_processor_count}")
    print(f"PyTorch import  : {time.time()-_t_torch:.1f}s")
except Exception as _exc:
    print(f"PyTorch unavailable: {_exc}")

# ── nvidia-smi usage metrics ───────────────────────────────────────────────
try:
    import nvidia_smi
    nvidia_smi.nvmlInit()
    for i in range(nvidia_smi.nvmlDeviceGetCount()):
        handle = nvidia_smi.nvmlDeviceGetHandleByIndex(i)
        name = nvidia_smi.nvmlDeviceGetName(handle)
        mem = nvidia_smi.nvmlDeviceGetMemoryInfo(handle)
        util = nvidia_smi.nvmlDeviceGetUtilizationRates(handle)
        usage_metrics["gpu"].append({
            "index": i,
            "name": name if isinstance(name, str) else name.decode(),
            "total_mb": mem.total // (1024 ** 2),
            "free_mb": mem.free // (1024 ** 2),
            "gpu_util_pct": util.gpu,
        })
    print(f"\nGPU(s) found: {usage_metrics['gpu']}")
except Exception as exc:
    print(f"nvidia-smi unavailable: {exc}")
    if accelerator == "cuda":
        print("WARNING: accelerator=cuda but no GPU found; falling back to cpu")
        accelerator = "cpu"

print(f"\n[T+{_elapsed()}] GPU check complete")


## Download DICOM from IDC

In [ ]:
DICOM_DIR = Path("/tmp/dicom")
DICOM_DIR.mkdir(parents=True, exist_ok=True)

client = IDCClient()
download_errors = []

print(f"[T+{_elapsed()}] Starting download of {len(series_uids)} series")

for uid in series_uids:
    dest = DICOM_DIR / uid
    dest.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    print(f"  Downloading {uid} ...", flush=True)
    try:
        client.download_from_selection(
            downloadDir=str(dest),
            seriesInstanceUID=uid,
        )
        elapsed = round(time.time() - t0, 1)
        dcm_files = list(dest.rglob("*.dcm"))
        size_mb = _dir_size_mb(dest)
        throughput = size_mb / elapsed if elapsed > 0 else 0
        usage_metrics["series"].setdefault(uid, {})["download_s"] = elapsed
        usage_metrics["series"][uid]["download_dcm_files"] = len(dcm_files)
        usage_metrics["series"][uid]["download_mb"] = round(size_mb, 1)
        print(f"  Downloaded {uid}: {len(dcm_files)} DICOM files  "
              f"{size_mb:.1f} MB  in {elapsed}s  ({throughput:.1f} MB/s)")

        # ── DICOM image-dimension metadata ────────────────────────────────
        rows, cols = None, None
        try:
            import pydicom
            if dcm_files:
                _ds = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True)
                rows = int(getattr(_ds, "Rows", 0)) or None
                cols = int(getattr(_ds, "Columns", 0)) or None
        except Exception:
            pass
        n_slices = len(dcm_files)
        total_pixels = rows * cols * n_slices if (rows and cols) else None
        usage_metrics["series"][uid]["slices"] = n_slices
        usage_metrics["series"][uid]["rows"] = rows
        usage_metrics["series"][uid]["cols"] = cols
        usage_metrics["series"][uid]["total_pixels"] = total_pixels
        _dim = f"{rows}x{cols}" if (rows and cols) else "?x?"
        _pix = f"  |  total pixels {total_pixels:,}" if total_pixels else ""
        print(f"    Dimensions  : {_dim} px  |  {n_slices} slices{_pix}")

    except Exception as exc:
        msg = f"{uid}: {traceback.format_exc()}"
        download_errors.append(msg)
        print(f"  ERROR downloading {uid}: {exc}")

if download_errors:
    Path("download_error_file.txt").write_text("\n".join(download_errors))

print(f"\n[T+{_elapsed()}] Download phase complete  "
      f"({len(series_uids)-len(download_errors)}/{len(series_uids)} succeeded)")


## Convert DICOM -> NIfTI with dcm2niix

In [ ]:
NIFTI_DIR = Path("/tmp/nifti")
NIFTI_DIR.mkdir(parents=True, exist_ok=True)

dcm2niix_errors = []
converted_uids = []
failed_download_uids = {e.split(":")[0] for e in download_errors}

print(f"[T+{_elapsed()}] Starting dcm2niix conversion for "
      f"{len(series_uids)-len(failed_download_uids)} series")

for uid in series_uids:
    if uid in failed_download_uids:
        continue

    dcm_path = DICOM_DIR / uid
    nii_path = NIFTI_DIR / uid
    nii_path.mkdir(parents=True, exist_ok=True)

    dcm_files = list(dcm_path.rglob("*.dcm"))
    print(f"  Converting {uid}  ({len(dcm_files)} DICOMs) ...", flush=True)
    t0 = time.time()
    result = subprocess.run(
        ["dcm2niix", "-z", "y", "-f", "%i_%s", "-o", str(nii_path), str(dcm_path)],
        capture_output=True,
        text=True,
    )
    elapsed = round(time.time() - t0, 1)

    nii_files = list(nii_path.glob("*.nii.gz"))
    if result.returncode != 0 or not nii_files:
        msg = f"{uid}: return_code={result.returncode}\n{result.stderr}"
        dcm2niix_errors.append(msg)
        print(f"  ERROR converting {uid} (return={result.returncode})")
        if result.stderr:
            print(f"    stderr: {result.stderr[:500]}")
    else:
        nii_sizes = {f.name: round(f.stat().st_size / (1024**2), 1) for f in nii_files}
        usage_metrics["series"].setdefault(uid, {})["dcm2niix_s"] = elapsed
        usage_metrics["series"][uid]["nifti_files_mb"] = nii_sizes
        converted_uids.append(uid)
        print(f"  Converted  {uid}: {elapsed}s  ->  "
              + ", ".join(f"{n} ({s} MB)" for n, s in nii_sizes.items()))

if dcm2niix_errors:
    Path("dcm2niix_error_file.txt").write_text("\n".join(dcm2niix_errors))

print(f"\n[T+{_elapsed()}] dcm2niix complete  "
      f"({len(converted_uids)}/{len(series_uids)-len(failed_download_uids)} succeeded)")


## Run MOOSE inference

moosez writes to `<MOOSE_OUT_DIR>/<uid>/moosez-<model>-<timestamp>/segmentations/` for each
requested model. The whole tree is compressed into a single archive below.

In [ ]:

# ── pre-import moosez / SimpleITK so import time is visible separately ────
print(f"[T+{_elapsed()}] Importing moosez and SimpleITK ...", flush=True)
_t_import = time.time()
from moosez import moose
import SimpleITK
from moosez import image_processing as _ip
print(f"[T+{_elapsed()}] moosez + SimpleITK imported in {time.time()-_t_import:.1f}s")

# ── helper: sample GPU memory via nvidia-smi ──────────────────────────────
def _gpu_mem_snapshot(label: str) -> dict:
    snap = {"label": label, "t": round(time.time() - NOTEBOOK_START, 1)}
    try:
        import nvidia_smi
        for i in range(nvidia_smi.nvmlDeviceGetCount()):
            h = nvidia_smi.nvmlDeviceGetHandleByIndex(i)
            m = nvidia_smi.nvmlDeviceGetMemoryInfo(h)
            u = nvidia_smi.nvmlDeviceGetUtilizationRates(h)
            snap[f"gpu{i}_used_mb"] = m.used // (1024**2)
            snap[f"gpu{i}_free_mb"] = m.free // (1024**2)
            snap[f"gpu{i}_util_pct"] = u.gpu
    except Exception:
        pass
    return snap

# ── helper: log a GPU snapshot to stdout and append to metrics list ───────
_gpu_snapshots = []
def _log_gpu(label: str):
    s = _gpu_mem_snapshot(label)
    _gpu_snapshots.append(s)
    parts = [f"T+{s['t']}s  {label}"]
    for k, v in s.items():
        if k.startswith("gpu"):
            parts.append(f"{k}={v}")
    print("  [GPU] " + "  ".join(parts), flush=True)

# ── MOOSE inference loop ──────────────────────────────────────────────────
MOOSE_OUT_DIR = Path("/tmp/moose_output")
MOOSE_OUT_DIR.mkdir(parents=True, exist_ok=True)

moose_errors = []

print(f"\n[T+{_elapsed()}] Starting MOOSE inference for {len(converted_uids)} series  "
      f"({len(models)} models each)\n")

for uid in converted_uids:
    nii_dir = NIFTI_DIR / uid
    out_path = MOOSE_OUT_DIR / uid
    out_path.mkdir(parents=True, exist_ok=True)

    nii_files = list(nii_dir.glob("*.nii.gz"))
    if not nii_files:
        msg = f"{uid}: no NIfTI files found in {nii_dir}"
        moose_errors.append(msg)
        print(f"ERROR: {msg}")
        continue
    nii_file = nii_files[0]
    nii_size_mb = round(nii_file.stat().st_size / (1024**2), 1)

    print(f"[T+{_elapsed()}] Series {uid}")
    print(f"  Input NIfTI : {nii_file.name}  ({nii_size_mb} MB)")
    _log_gpu("before-series")

    series_start = time.time()
    series_model_times = {}

    stats_dir = out_path / "stats"
    stats_dir.mkdir(exist_ok=True)

    for model_idx, model in enumerate(models):
        print(f"\n  [{model_idx+1}/{len(models)}] {model}  T+{_elapsed()}", flush=True)
        _log_gpu(f"before-{model}")

        t0 = time.time()
        try:
            seg_paths, model_objs = moose(str(nii_file), [model], str(out_path), accelerator)
            inference_elapsed = round(time.time() - t0, 1)
            series_model_times[model] = inference_elapsed

            seg_files = list(out_path.rglob("*.nii.gz"))
            seg_sizes = {Path(f).name: round(Path(f).stat().st_size/(1024**2), 1)
                         for f in (seg_paths or [])}
            _log_gpu(f"after-{model}")
            print(f"  inference : {inference_elapsed}s  |  "
                  f"seg files so far: {len(seg_files)}  |  "
                  f"new segs: {seg_sizes}")

            # ── stats CSVs ────────────────────────────────────────────────
            if seg_paths and model_objs:
                t_stats = time.time()
                try:
                    model_obj = model_objs[0]
                    seg_image = SimpleITK.ReadImage(str(seg_paths[0]))
                    raw_image = SimpleITK.ReadImage(str(nii_file))

                    vol_csv = str(stats_dir / f"{model_obj.multilabel_prefix}{uid}_volume.csv")
                    _ip.get_shape_statistics(seg_image, model_obj, vol_csv)

                    if seg_image.GetSize() != raw_image.GetSize():
                        resampler = SimpleITK.ResampleImageFilter()
                        resampler.SetReferenceImage(raw_image)
                        resampler.SetInterpolator(SimpleITK.sitkNearestNeighbor)
                        seg_image = resampler.Execute(seg_image)
                    int_csv = str(stats_dir / f"{model_obj.multilabel_prefix}{uid}_CT_intensity.csv")
                    _ip.get_intensity_statistics(raw_image, seg_image, model_obj, int_csv)

                    stats_elapsed = round(time.time() - t_stats, 1)
                    print(f"  stats     : {stats_elapsed}s  |  "
                          f"{Path(vol_csv).name}, {Path(int_csv).name}")
                    series_model_times[f"{model}_stats_s"] = stats_elapsed
                except Exception as stats_exc:
                    print(f"  WARNING: stats failed for {model}: {stats_exc}")

        except Exception as exc:
            msg = f"{uid}/{model}: {traceback.format_exc()}"
            moose_errors.append(msg)
            print(f"  ERROR {model}: {exc}")

    total_elapsed = round(time.time() - series_start, 1)
    usage_metrics["series"].setdefault(uid, {})["moose_s"] = total_elapsed
    usage_metrics["series"][uid]["moose_models_s"] = series_model_times
    usage_metrics["series"][uid]["gpu_snapshots"] = _gpu_snapshots.copy()
    _gpu_snapshots.clear()

    _log_gpu("after-series")
    print(f"\n[T+{_elapsed()}] Series {uid} done: "
          f"{total_elapsed}s total  ({len(series_model_times)} model(s))")

if moose_errors:
    Path("moose_errors.txt").write_text("\n".join(moose_errors))

print(f"\n[T+{_elapsed()}] MOOSE inference phase complete  "
      f"({len(converted_uids)-len(moose_errors)}/{len(converted_uids)} series succeeded)")


## Package outputs

In [ ]:
import glob as _glob

print(f"[T+{_elapsed()}] Starting output packaging")


def compress_dir(src_dir: Path, out_file: str) -> None:
    """Tar a directory and compress with lz4."""
    t0 = time.time()
    src_raw_mb = _dir_size_mb(src_dir)
    print(f"  Compressing {src_dir.name}  ({src_raw_mb:.1f} MB uncompressed) ...", flush=True)
    cmd = f"tar -cf - -C {src_dir.parent} {src_dir.name} | lz4 > {out_file}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Compression failed: {result.stderr}")
    elapsed = round(time.time() - t0, 1)
    size_mb = Path(out_file).stat().st_size / (1024 ** 2)
    ratio = src_raw_mb / size_mb if size_mb > 0 else 0
    print(f"  Created {out_file}  ({size_mb:.1f} MB compressed  ratio {ratio:.1f}x  in {elapsed}s)")


# Extract organ indices and bundle into archive
try:
    from moosez.models import MODEL_METADATA
    from moosez import system as _system

    _folder_to_model = {meta["folder_name"]: name for name, meta in MODEL_METADATA.items()}
    organ_indices = {}
    for _json_path in _glob.glob(f"{_system.MODELS_DIRECTORY_PATH}/**/dataset.json", recursive=True):
        _folder = os.path.relpath(_json_path, _system.MODELS_DIRECTORY_PATH).split(os.sep)[0]
        _model_name = _folder_to_model.get(_folder)
        if _model_name is None:
            continue
        with open(_json_path) as _f:
            _labels = json.load(_f).get("labels", {})
        organ_indices[_model_name] = {
            int(k): v for k, v in _labels.items()
            if k.isdigit() and v.lower() not in ("background", "")
        }
    (MOOSE_OUT_DIR / "moose_organ_indices.json").write_text(json.dumps(organ_indices, indent=2))
    print(f"  Bundled organ indices for {len(organ_indices)} model(s): {list(organ_indices)}")
except Exception as _exc:
    print(f"  WARNING: could not extract organ indices: {_exc}")


compress_dir(MOOSE_OUT_DIR, "moose_segmentations.tar.lz4")
print(f"\n[T+{_elapsed()}] Packaging complete")


## Write usage metrics

In [ ]:
usage_metrics["total_elapsed_s"] = round(time.time() - NOTEBOOK_START, 1)

metrics_json = json.dumps(usage_metrics, indent=2)
metrics_path = Path("moose_inference_UsageMetrics.json")
metrics_path.write_text(metrics_json)

subprocess.run(
    f"lz4 -f {metrics_path} moose_inference_UsageMetrics.lz4",
    shell=True,
    check=True,
)

print(metrics_json)
print(f"\n[T+{_elapsed()}] Usage metrics written")


## Summary

In [ ]:
total_s = round(time.time() - NOTEBOOK_START, 1)

print("=" * 70)
print("MOOSE Inference Summary")
print("=" * 70)
print(f"  Total wall-clock time : {total_s}s")
print(f"  Total series          : {len(series_uids)}")
print(f"  Download failures     : {len(download_errors)}")
print(f"  dcm2niix failures     : {len(dcm2niix_errors)}")
print(f"  MOOSE failures        : {len(moose_errors)}")
print(f"  Models used           : {models}")
print()

# ── per-series phase breakdown ─────────────────────────────────────────────
for uid, metrics in usage_metrics.get("series", {}).items():
    print(f"  {uid}")
    dl_s   = metrics.get("download_s", 0)
    dl_mb  = metrics.get("download_mb", "?")
    dcm_n  = metrics.get("download_dcm_files", "?")
    conv_s = metrics.get("dcm2niix_s", 0)
    moose_s = metrics.get("moose_s", 0)

    rows   = metrics.get("rows")
    cols   = metrics.get("cols")
    slices = metrics.get("slices")
    total_pixels = metrics.get("total_pixels")
    _dim   = f"{rows}x{cols}" if (rows and cols) else "?x?"
    _pix   = f"  total pixels {total_pixels:,}" if total_pixels else ""
    _slices = f"{slices}" if slices is not None else "?"
    print(f"    dimensions  : {_dim} px  |  {_slices} slices{_pix}")
    print(f"    download    : {dl_s:>7.1f}s  ({dcm_n} DICOM files  {dl_mb} MB)")
    print(f"    dcm2niix    : {conv_s:>7.1f}s")

    model_times = metrics.get("moose_models_s", {})
    inference_total = 0
    stats_total = 0
    for m in models:
        inf_s   = model_times.get(m, 0)
        stat_s  = model_times.get(f"{m}_stats_s", 0)
        inference_total += inf_s
        stats_total += stat_s
        print(f"    {m:<35}  infer {inf_s:>6.1f}s  stats {stat_s:>5.1f}s")

    print(f"    {'inference subtotal':<35}  {inference_total:>7.1f}s")
    print(f"    {'stats subtotal':<35}  {stats_total:>7.1f}s")
    print(f"    {'moose total (all models)':<35}  {moose_s:>7.1f}s")
    print()

# ── cohort totals ──────────────────────────────────────────────────────────
_all_series = usage_metrics.get("series", {})
_cohort_dcm   = [m.get("download_dcm_files", 0) for m in _all_series.values() if m.get("download_dcm_files") is not None]
_cohort_mb    = [m.get("download_mb", 0)         for m in _all_series.values() if m.get("download_mb") is not None]
_cohort_sl    = [m.get("slices", 0)              for m in _all_series.values() if m.get("slices") is not None]
_cohort_pix   = [m.get("total_pixels", 0)        for m in _all_series.values() if m.get("total_pixels") is not None]

if _all_series:
    print("  Cohort totals:")
    print(f"    Series                : {len(_all_series)}")
    if _cohort_dcm:
        print(f"    DICOM files total     : {sum(_cohort_dcm):,}  "
              f"(min {min(_cohort_dcm):,}  max {max(_cohort_dcm):,}  "
              f"avg {sum(_cohort_dcm)//len(_cohort_dcm):,} per series)")
    if _cohort_mb:
        print(f"    Download size total   : {sum(_cohort_mb):.1f} MB  "
              f"(min {min(_cohort_mb):.1f}  max {max(_cohort_mb):.1f}  "
              f"avg {sum(_cohort_mb)/len(_cohort_mb):.1f} MB per series)")
    if _cohort_sl:
        print(f"    Slices total          : {sum(_cohort_sl):,}  "
              f"(min {min(_cohort_sl):,}  max {max(_cohort_sl):,}  "
              f"avg {sum(_cohort_sl)//len(_cohort_sl):,} per series)")
    if _cohort_pix:
        print(f"    Total pixels (all)    : {sum(_cohort_pix):,}  "
              f"(min {min(_cohort_pix):,}  max {max(_cohort_pix):,}  "
              f"avg {sum(_cohort_pix)//len(_cohort_pix):,} per series)")
    print()

# ── phase share of total ───────────────────────────────────────────────────
if total_s > 0:
    all_dl   = sum(m.get("download_s", 0) for m in usage_metrics.get("series", {}).values())
    all_conv = sum(m.get("dcm2niix_s", 0) for m in usage_metrics.get("series", {}).values())
    all_moose= sum(m.get("moose_s", 0) for m in usage_metrics.get("series", {}).values())
    other    = total_s - all_dl - all_conv - all_moose
    print("  Phase share of total wall-clock:")
    for label, secs in [("download", all_dl), ("dcm2niix", all_conv),
                        ("moose", all_moose), ("overhead/other", other)]:
        pct = 100 * secs / total_s
        bar = "#" * int(pct / 2)
        print(f"    {label:<20} {secs:>7.1f}s  {pct:>5.1f}%  {bar}")

print("=" * 70)

if len(moose_errors) == len(series_uids):
    raise RuntimeError("All series failed MOOSE inference - see moose_errors.txt")
